# IndustrialGuard AI ? PaDiM on MPDD `bracket_black`

This notebook prepares and runs exactly one full benchmark: PaDiM, MPDD `bracket_black`, ResNet18 ImageNet pretrained weights, seed 42, 256?256, CUDA. It reuses the repository pipeline and refuses to train unless the verified split, resources, GPU, and weights are ready. It does not run a smoke cap or other model/category.

Google Drive mount, repository ZIP upload, and pretrained-weight download each require an explicit user action.


## A. Colab environment

In [ ]:
import os, sys, shutil, subprocess, platform, json, time, hashlib, zipfile, threading, queue
from pathlib import Path

import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this benchmark. Select a GPU runtime in Runtime > Change runtime type, then rerun.")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}; memory={props.total_memory / 1024**3:.2f} GiB; capability={props.major}.{props.minor}")
free = shutil.disk_usage("/content").free
print(f"Available /content disk: {free / 1024**3:.2f} GiB")
subprocess.run(["nvidia-smi"], check=True)


## B. Project source

In [ ]:
# Leave blank when there is no configured official remote. Do not invent a URL.
PROJECT_GIT_URL = ""
PROJECT_ROOT = Path("/content/industrialguard-ai")

if PROJECT_GIT_URL.strip():
    if PROJECT_ROOT.exists():
        raise FileExistsError(f"Refusing to overwrite existing project directory: {PROJECT_ROOT}")
    subprocess.run(["git", "clone", PROJECT_GIT_URL, str(PROJECT_ROOT)], check=True)
else:
    from google.colab import files
    print("Upload a source ZIP built from the project, excluding .git, .venv, .cache, data, datasets, checkpoints, runs, and MLflow databases.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one project source ZIP.")
    source_zip = Path("/content") / zip_names[0]
    forbidden = (".git/", ".venv/", ".cache/", "cache/", "datasets/", "data/raw/", "data/processed/", "checkpoints/", "runs/", "mlflow.db")
    with zipfile.ZipFile(source_zip) as zf:
        bad = [n for n in zf.namelist() if any(part in n.replace("\\", "/").lower() for part in forbidden)]
        if bad:
            raise RuntimeError(f"Source ZIP contains excluded runtime/data files: {bad[:5]}")
        for member in zf.infolist():
            destination = (Path("/content/project-source") / member.filename).resolve()
            if not destination.is_relative_to(Path("/content/project-source").resolve()):
                raise RuntimeError(f"Unsafe ZIP member path: {member.filename}")
        zf.extractall("/content/project-source")
    candidates = [p for p in Path("/content/project-source").rglob("pyproject.toml") if (p.parent / "src/industrialguard").is_dir()]
    if len(candidates) != 1:
        raise RuntimeError("ZIP must contain exactly one IndustrialGuard project root with pyproject.toml and src/industrialguard/.")
    PROJECT_ROOT = candidates[0].parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
required = ["pyproject.toml", "configs/dataset.yaml", "configs/models/padim.yaml", "configs/benchmark.yaml", "reports/dataset_audit/mpdd_inventory.csv", "reports/model_development/experimental_protocol.md", "scripts/run_benchmark.py"]
missing = [name for name in required if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Source ZIP is missing required project/DVC payload files: {missing}")
print("Project root:", PROJECT_ROOT)
print("Git remote configuration is not required for source ZIP mode.")


## C. Google Drive and MPDD archive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")  # Google prompts for explicit account authorization.

DRIVE_ROOT = Path("/content/drive/MyDrive/IndustrialGuardAI")
DRIVE_ARCHIVE = DRIVE_ROOT / "datasets/MPDD (1).zip"
LOCAL_ARCHIVE = Path("/content/MPDD (1).zip")
CACHE_ROOT = Path("/content/industrialguard_cache")
# The runner uses the configured relative .cache/datasets path; point that path to Colab local storage.
project_cache = PROJECT_ROOT / ".cache"
project_cache.mkdir(exist_ok=True)
cache_link = project_cache / "datasets"
if cache_link.exists() and not cache_link.is_symlink():
    raise RuntimeError(f"Refusing to reuse unexpected existing cache path: {cache_link}")
if not cache_link.exists():
    cache_link.symlink_to(CACHE_ROOT, target_is_directory=True)
EXPORT_ROOT = DRIVE_ROOT / "artifacts/experiments"
if not DRIVE_ARCHIVE.is_file() or not os.access(DRIVE_ARCHIVE, os.R_OK):
    raise FileNotFoundError(f"Readable MPDD archive required at {DRIVE_ARCHIVE}")
archive_bytes = DRIVE_ARCHIVE.stat().st_size
# Reserve archive size plus 3 GiB headroom for selected verified cache entries and runtime files.
free = shutil.disk_usage("/content").free
if free < archive_bytes + 3 * 1024**3:
    raise OSError(f"Insufficient /content space: need archive size + 3 GiB; have {free / 1024**3:.2f} GiB free.")
shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
source_sha = sha256_file(DRIVE_ARCHIVE)
local_sha = sha256_file(LOCAL_ARCHIVE)
if source_sha != local_sha:
    raise OSError("Local archive copy SHA-256 differs from the Drive source.")
with zipfile.ZipFile(LOCAL_ARCHIVE) as archive:
    corrupt = archive.testzip()
    if corrupt is not None:
        raise OSError(f"MPDD archive CRC check failed at {corrupt}")
print("MPDD archive is readable; full ZIP CRC passed; SHA-256:", source_sha)
print("Original archive was read only; verified local copy:", LOCAL_ARCHIVE)
os.environ["INDUSTRIALGUARD_MPDD_ARCHIVE"] = str(LOCAL_ARCHIVE)
os.environ["TORCH_HOME"] = "/content/torch-cache"
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///" + (PROJECT_ROOT / "runs/mlflow/mlflow.db").as_posix()
os.environ["INDUSTRIALGUARD_CACHE_DIR"] = str(CACHE_ROOT)


## D. Dependencies and CLI inspection

In [ ]:
# Install the repository package without allowing pip to replace Colab's CUDA-enabled PyTorch stack.
# The constraints below come from pyproject.toml; anomalib and DVC are not needed for PaDiM.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torchvision>=0.25.0", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "Pillow>=10.0", "numpy>=1.24", "scikit-learn>=1.3", "scipy>=1.10", "mlflow>=3.0,<4"], check=True)
from packaging.version import Version
if Version(torch.__version__.split("+")[0]) < Version("2.2"):
    raise RuntimeError(f"Installed PyTorch {torch.__version__} does not meet pyproject.toml torch>=2.2; refusing to change the CUDA PyTorch stack automatically.")
import torchvision, numpy, sklearn, scipy, mlflow
print("Torchvision:", torchvision.__version__)
_compatibility_model = torchvision.models.resnet18(weights=None)  # construction-only compatibility check; no random weights used in training
del _compatibility_model
print("NumPy:", numpy.__version__, "scikit-learn:", sklearn.__version__, "SciPy:", scipy.__version__, "MLflow:", mlflow.__version__)
print("Torch/CUDA/Torchvision:", torch.__version__, torch.version.cuda, torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA became unavailable after dependency setup; stop here.")
subprocess.run([sys.executable, "scripts/run_benchmark.py", "--help"], check=True)


## E. Pretrained ResNet18 weight verification

In [ ]:
from torchvision.models import ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
weight_url = weights.url
weight_file = weight_url.rsplit("/", 1)[-1]
weight_path = Path(os.environ["TORCH_HOME"]) / "hub/checkpoints" / weight_file
print("Backbone: torchvision ResNet18")
print("Weight identifier:", str(weights))
print("Resource URL (for provenance):", weight_url)
print("Expected cached path:", weight_path)
ALLOW_WEIGHT_DOWNLOAD = False  # Change to True only to explicitly authorize this resource download.
if not weight_path.is_file():
    print("Missing pretrained weights. The runner must not be started yet.")
    print("To proceed, explicitly set ALLOW_WEIGHT_DOWNLOAD=True and rerun this cell; this downloads the exact torchvision DEFAULT checkpoint.")
    if ALLOW_WEIGHT_DOWNLOAD:
        weights.get_state_dict(progress=True, check_hash=True)
    else:
        raise FileNotFoundError("ResNet18 ImageNet weights are absent. No download was performed.")
weight_sha = sha256_file(weight_path)
if not weight_file.startswith("resnet18-") or not weight_file.endswith(".pth") or weight_sha[:8] not in weight_file:
    raise OSError(f"Weight file name/hash does not match torchvision's hash-prefix check: {weight_file}")
print("Verified weight SHA-256:", weight_sha)


## F. Protocol and full split preflight

In [ ]:
import csv
from industrialguard.data.split_manager import create_splits
from industrialguard.data.discovery import ImageRecord
from industrialguard.data.data_module import IndustrialDataModule

with open(PROJECT_ROOT / "reports/dataset_audit/mpdd_inventory.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if r["category"] == "bracket_black"]
records = [ImageRecord(dataset=r["dataset"], category=r["category"], image_path=r["image_path"], image_type=r["image_type"], split=r["split"], defect_type=r["defect_type"], mask_path=r["mask_path"] or None) for r in rows]
splits = create_splits(records, validation_fraction=0.1, seed=42)
counts = {k: len(v) for k, v in splits.items()}
test_normal = sum(r.image_type == "normal" for r in splits["test"])
test_anomalous = sum(r.image_type == "anomalous" for r in splits["test"])
mask_count = sum(bool(r.mask_path) for r in splits["test"])
print({"train":counts["train"], "validation_normal":counts["validation"], "test":counts["test"], "normal_test":test_normal, "anomalous_test":test_anomalous, "test_masks":mask_count})
expected = {"train":260, "validation":29, "test":79, "normal_test":32, "anomalous_test":47, "test_masks":47}
observed = {"train":counts["train"], "validation":counts["validation"], "test":counts["test"], "normal_test":test_normal, "anomalous_test":test_anomalous, "test_masks":mask_count}
if observed != expected:
    raise RuntimeError(f"Audited split mismatch: expected {expected}, observed {observed}; stop before training.")
if any(r.image_type != "normal" for r in splits["train"] + splits["validation"]):
    raise RuntimeError("Train/validation split contains anomalies; refusing to continue.")
if len({r.image_path for s in splits.values() for r in s}) != sum(map(len, splits.values())):
    raise RuntimeError("Data leakage detected between splits.")
# Materialize and SHA-256 verify only selected source members via the existing cache manager.
data = IndustrialDataModule("mpdd", ["bracket_black"], dataset_config="configs/dataset.yaml", experiment_config="configs/models/padim.yaml", cache_dir=CACHE_ROOT, image_size=256, batch_size=8, seed=42, validation_fraction=0.1)
data.setup()
actual = {k: len(v) for k, v in data.split_records.items()}
if actual != {"train":260,"validation":29,"test":79}:
    raise RuntimeError(f"Data module split mismatch: {actual}")
print("Archive-selected members are materialized and SHA-256 verified under", CACHE_ROOT)


## G. Experiment configuration and execution gate

In [ ]:
# Match configs/models/padim.yaml and configs/benchmark.yaml. No record caps are passed.
BENCHMARK_COMMAND = [sys.executable, "scripts/run_benchmark.py", "--dataset", "mpdd", "--model", "padim", "--category", "bracket_black", "--device", "cuda", "--batch-size", "8", "--image-size", "256", "--seed", "42", "--fail-fast"]
print("Prepared command:", " ".join(BENCHMARK_COMMAND))
print("Threshold method: 99.5th percentile of normal-only deterministic validation scores/maps; test labels/masks are excluded from threshold selection.")
print("Training data: official train/good only; validation: seed-42 deterministic 10% subset of train/good; test: unchanged official test split.")
print("Model: PaDiM; backbone: ResNet18 ImageNet DEFAULT; feature_dim=50; covariance_epsilon=1e-4; patch_batch_size=256; smoothing_sigma=2.0.")
RUN_BENCHMARK = False  # Explicitly change to True after reviewing all preflight cells.
if not torch.cuda.is_available() or not weight_path.is_file() or observed != expected:
    raise RuntimeError("Preflight incomplete; training is blocked.")
if not RUN_BENCHMARK:
    print("Prepared only. Set RUN_BENCHMARK=True to start this one benchmark.")


In [ ]:
# The only training cell. It runs one category/model; no --all-categories, caps, or download flag.
if not RUN_BENCHMARK:
    raise RuntimeError("Set RUN_BENCHMARK=True only after reviewing the preflight and authorizing execution.")

# Stream runner messages and sample GPU memory while the existing runner performs fitting/evaluation.
from collections import deque
messages = queue.Queue()
def reader_thread(pipe):
    for line in iter(pipe.readline, ""):
        messages.put(line.rstrip())
    pipe.close()
process = subprocess.Popen(BENCHMARK_COMMAND, cwd=PROJECT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=reader_thread, args=(process.stdout,), daemon=True).start()
gpu_samples = []
while process.poll() is None:
    try:
        while True:
            print(messages.get_nowait())
    except queue.Empty:
        pass
    if torch.cuda.is_available():
        query = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True, capture_output=True, check=True)
        used_mib = int(query.stdout.strip().splitlines()[0])
        gpu_samples.append({"timestamp":time.time(),"system_gpu_memory_used_mib":used_mib})
    time.sleep(5)
try:
    while True:
        print(messages.get_nowait())
except queue.Empty:
    pass
if process.returncode != 0:
    raise RuntimeError(f"Benchmark runner failed with exit code {process.returncode}; inspect runs/benchmark/*/*/train.log and status.json. No metrics should be claimed.")


## H. MLflow retrieval and Drive artifact export

In [ ]:
# The runner uses its existing local SQLite MLflow setup. Do not place SQLite on Drive.
from mlflow.tracking import MlflowClient
tracking_uri = "sqlite:///" + (PROJECT_ROOT / "runs/mlflow/mlflow.db").as_posix()
mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient(tracking_uri=tracking_uri)
experiment = client.get_experiment_by_name("IndustrialGuard-Phase4-Benchmark")
if experiment is None:
    raise RuntimeError("Expected local MLflow experiment was not created.")
runs_found = client.search_runs([experiment.experiment_id], filter_string="tags.run_role = 'benchmark_model_category' and tags.dataset = 'mpdd' and tags.category = 'bracket_black' and tags.model = 'padim'", order_by=["attributes.start_time DESC"], max_results=20)
completed = [r for r in runs_found if r.data.tags.get("experiment_state") != "failed" and r.data.tags.get("run_role") == "benchmark_model_category"]
if not completed:
    # completed status is authoritative when the child tag was not set by this runner version
    candidates = list((PROJECT_ROOT / "runs/benchmark").glob("*/attempt-*/status.json"))
    completed_status = [json.loads(p.read_text()) for p in candidates if json.loads(p.read_text()).get("state") == "completed" and json.loads(p.read_text()).get("dataset") == "mpdd" and json.loads(p.read_text()).get("category") == "bracket_black" and json.loads(p.read_text()).get("model") == "padim"]
    if not completed_status:
        raise RuntimeError("No completed experiment status found; stop without exporting benchmark claims.")
    chosen_status = completed_status[-1]
    run_id = chosen_status["mlflow_run_id"]
    exp_id = chosen_status["experiment_id"]
else:
    chosen_run = completed[0]
    run_id = chosen_run.info.run_id
    exp_id = chosen_run.data.tags["experiment_id"]
run = client.get_run(run_id)
print("Verified retrievable MLflow run:", run_id, "experiment ID:", exp_id)
print("Tracking database (runtime-local):", tracking_uri)

export_dir = EXPORT_ROOT / exp_id
export_dir.mkdir(parents=True, exist_ok=False)  # Never overwrite a prior export.
attempt_files = list((PROJECT_ROOT / "runs/benchmark" / exp_id).glob("attempt-*/status.json"))
completed_attempt = next(p.parent for p in reversed(attempt_files) if json.loads(p.read_text()).get("state") == "completed")
chosen_status = json.loads((completed_attempt / "status.json").read_text())
shutil.copytree(completed_attempt, export_dir / "attempt")
mlflow_artifacts = mlflow.artifacts.download_artifacts(run_id=run_id)
shutil.copytree(mlflow_artifacts, export_dir / "mlflow_artifacts")
run_summary = {"run_id":run_id,"experiment_id":exp_id,"tracking_uri_runtime_local":tracking_uri,"params":run.data.params,"metrics":run.data.metrics,"tags":run.data.tags,"artifact_export":"mlflow_artifacts/","tracking_database_exported":False,"note":"The SQLite tracking database remains in the temporary Colab runtime; portable run artifacts and summary are exported to Drive."}
(export_dir / "mlflow_run_summary.json").write_text(json.dumps(run_summary, indent=2, sort_keys=True, default=str))
provenance = {
    "experiment_id":exp_id, "mlflow_run_id":run_id, "model":"padim", "model_version":"1",
    "dataset":"mpdd", "category":"bracket_black", "backbone":"torchvision ResNet18",
    "pretrained_weight_identifier":str(weights), "pretrained_weight_url":weight_url, "pretrained_weight_sha256":weight_sha,
    "mpdd_archive_sha256":source_sha, "input_resolution":256, "seed":42, "batch_size":8,
    "expected_split_counts":expected, "observed_split_counts":observed,
    "threshold_method":"normal-only validation 99.5th percentile",
    "python":platform.python_version(), "torch":torch.__version__, "torchvision":torchvision.__version__,
    "torch_cuda_runtime":torch.version.cuda, "gpu_name":torch.cuda.get_device_name(0),
    "gpu_total_memory_bytes":torch.cuda.get_device_properties(0).total_memory,
    "training_duration_seconds":chosen_status.get("training_duration_seconds"),
    "benchmark_command":BENCHMARK_COMMAND,
}
(export_dir / "reproducibility_metadata.json").write_text(json.dumps(provenance, indent=2, sort_keys=True, default=str))
(export_dir / "gpu_memory_samples.json").write_text(json.dumps(gpu_samples, indent=2))
(export_dir / "gpu_memory_peak.json").write_text(json.dumps({"sampling_interval_seconds":5,"system_peak_used_mib":max((x["system_gpu_memory_used_mib"] for x in gpu_samples), default=None)}, indent=2))
# Export all per-test sample scores, predicted maps, and transformed ground-truth masks using the fitted checkpoint.
import numpy as np
from industrialguard.models.registry import create_model
from industrialguard.mlops.reproducibility import seed_everything
seed_everything(42)
config_path = completed_attempt / "config.json"
config = json.loads(config_path.read_text())
model = create_model("padim", **config["hyperparameters"], backbone_name="resnet18", pretrained=False, seed=42, device="cuda")
model.load_state(completed_attempt / "checkpoints/padim/mpdd/bracket_black/model.pt", map_location="cuda")
model.eval()
prediction_dir = export_dir / "predictions"
prediction_dir.mkdir()
with torch.inference_mode():
    for batch in data.test_dataloader():
        output = model.predict(batch["image"].to("cuda"))
        for i, sample_id in enumerate(batch["sample_id"]):
            filename = hashlib.sha256(sample_id.encode()).hexdigest()[:20] + ".npz"
            np.savez_compressed(prediction_dir / filename, sample_id=np.array(sample_id), image_path=np.array(batch["image_path"][i]), label=int(batch["label"][i]), image_score=float(output.image_score[i].cpu()), anomaly_map=output.anomaly_map[i,0].cpu().numpy().astype(np.float32), ground_truth_mask=batch["mask"][i,0].cpu().numpy().astype(np.uint8))
# Existing evaluator images are actual predictions; verify representative original/mask/map images were logged.
repr_dir = export_dir / "mlflow_artifacts/representative_images"
print("Representative prediction visualizations:", repr_dir)
for f in sorted(export_dir.rglob("*")):
    if f.is_file():
        print(f.relative_to(export_dir), f.stat().st_size)
assert any(export_dir.rglob("model.pt"))
assert any(export_dir.rglob("bracket_black.json"))
assert len(list(prediction_dir.glob("*.npz"))) == 79
# Build actual-prediction comparison panels (normal and anomalous) including overlays.
from PIL import Image, ImageDraw
visual_dir = export_dir / "visualizations"
visual_dir.mkdir()
for kind in ("normal", "anomalous"):
    prefix = f"bracket_black_{kind}"
    image = Image.open(repr_dir / f"{prefix}_image.png").convert("RGB")
    mask = Image.open(repr_dir / f"{prefix}_mask.png").convert("L")
    heat = Image.open(repr_dir / f"{prefix}_predicted_map.png").convert("L")
    red = Image.new("RGB", image.size, (255, 30, 0))
    alpha = heat.point(lambda x: int(x * 0.55))
    overlay = Image.composite(red, image, alpha)
    panels = [image, mask.convert("RGB"), heat.convert("RGB"), overlay]
    labels = ["Original", "Ground truth", "Predicted heatmap", "Prediction overlay"]
    canvas = Image.new("RGB", (image.width * 2, (image.height + 28) * 2), "white")
    draw = ImageDraw.Draw(canvas)
    for index, (panel, label) in enumerate(zip(panels, labels)):
        x = (index % 2) * image.width
        y = (index // 2) * (image.height + 28)
        canvas.paste(panel, (x, y + 28))
        draw.text((x + 4, y + 6), label, fill="black")
    canvas.save(visual_dir / f"{prefix}_comparison.png")
for f in sorted(export_dir.rglob("*")):
    if f.is_file():
        print(f.relative_to(export_dir), f.stat().st_size)
print("Drive export verified:", export_dir)


## I. Local Windows restore

Download the experiment folder from `MyDrive/IndustrialGuardAI/artifacts/experiments/<experiment-id>/` to Windows. Read `attempt/metrics/bracket_black.json`, `attempt/config.json`, `attempt/environment.json`, and `mlflow_run_summary.json`; inspect PNGs under `mlflow_artifacts/representative_images/`; and load the checkpoint with the project model factory and `model.load_state(checkpoint, map_location="cpu")`. For inference, reconstruct PaDiM with the recorded `feature_dim`, covariance, smoothing, and seed values, `backbone_name="resnet18"`, `pretrained=False`, then call `predict` on a 256?256 ImageNet-normalized tensor. The checkpoint includes the fitted backbone state and PaDiM statistics, so loading it does not download weights. The runtime SQLite database is not exported; the Drive run summary and complete MLflow artifact bundle are the durable tracking export. Keep downloaded checkpoints and tracking artifacts outside Git.
